### Introduction

This is a notebook to help with data manipulation. It contains both the feature engineering and data manipulation code necessary for this RecSys project.
The data is from a VNDB dump, using the open Database license.



### Information

The data is split 85/15 into training and testing sets. The training set is used to train the model and the testing set is used to evaluate the model.

Data is split on a ratio and timestamp basis. THe first 85 percent of a user's reviews are used to train the model and the last 15 percent are used to test the model. This is to ensure there is no temportal leakage.




In [2]:
import os
from dotenv import load_dotenv
import pandas as pd
import torch
from torch.utils.data import Dataset
from sqlalchemy import create_engine
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import parquet
from collections import defaultdict

In [3]:
load_dotenv()
db_uri = os.getenv('DB_URI')
engine = create_engine(db_uri)
query = """
SELECT * FROM votes
"""

df = pd.read_sql(query, engine)

In [4]:
#Create database matching users to their number of reviews. 

user_counts = df['user_id'].value_counts()

In [5]:
#We use a k-core of 10 to remove users with less than 10 ratings, this is a commonly used technique in recommender systems. 
#Since users are the index of this dataframe this will return a list of users that satisfy this condition.
dense_users = user_counts[user_counts >= 10].index

# This dataset now only contains users who can be stably trained and tested, as they have more than 10 ratings, mitigating the cold start problem.
df_clean = df[df['user_id'].isin(dense_users)].copy()

df_clean = df_clean.sort_values(by=['user_id', 'date'])

#Calculate the 85th percentile cap.
cap = int(df["user_id"].value_counts().quantile(0.85))

#Restrict power users from over influencing the dataset by capping the number of ratings a user can have.
capped_df = df.groupby("user_id").tail(cap)


In [6]:
#Function to split a single user's history.
def split_user_data(group, test_ratio=0.15):
    num_test = int(np.ceil(len(group) * test_ratio))
    # Safety check: if user only has 1 or 2 interactions, keep them in train
    train = group.iloc[:-num_test]
    test = group.iloc[-num_test:]
    return train, test

#Apply across all users.
train_slices = []
test_slices = []

for user_id, group in capped_df.groupby('user_id'):
    train, test = split_user_data(group)
    train_slices.append(train)
    test_slices.append(test)

train_set = pd.concat(train_slices)
test_set = pd.concat(test_slices)

#Final Safety Filter: Drop test samples if the user somehow didn't make it to train
test_set = test_set[test_set['user_id'].isin(train_set['user_id'])]

In [7]:

unique_users = train_set['user_id'].unique()
unique_vns= set(train_set['vn_id']).union(set(test_set['vn_id']))

#Map users and vns to contiguous integers starting at 1, since we will use 0 for null/missing data 
user_to_idx = {u:i for i,u in enumerate(unique_users, start=1)}
vn_to_idx ={v:i for i,v in enumerate(unique_vns, start=1)}

train_set['user_idx'] = train_set['user_id'].map(user_to_idx)
train_set['vn_idx'] = train_set['vn_id'].map(vn_to_idx)
test_set['user_idx'] = test_set['user_id'].map(user_to_idx)
test_set['vn_idx'] = test_set['vn_id'].map(vn_to_idx)


In [8]:
query = """
SELECT * FROM tags_vn
"""

tags_df = pd.read_sql(query, engine)

In [9]:
tags_df.head()

unique_tags = tags_df['tag'].unique()

tag_to_idx = {tag: idx for idx, tag in enumerate(unique_tags, start=1)}

In [10]:
#Drops the Nan values from the series and returns the mean of the series, used so the average of only the actual spoiler votes is taken
def mean_without_nan(x):
    clean_series = x.dropna()
    return clean_series.mean() if not clean_series.empty else np.nan

In [11]:
grouped_data = tags_df.groupby(['vid', 'tag']).agg(
    weight=('vote', 'mean'),
    spoiler_score=('spoiler', mean_without_nan)).reset_index()


In [12]:

grouped_data['spoiler_score'] = grouped_data['spoiler_score'].fillna(0)


In [13]:

grouped_means = tags_df.groupby(['vid', 'tag'])['vote'].mean()

vn_idx_to_tags = defaultdict(dict)

for _, data in grouped_data.iterrows():
    #If vid not in vn_to_idx, skip, as it is not rated by users in train or test set
    if data['vid'] in vn_to_idx:
        idx = vn_to_idx[data['vid']] 
        vn_idx_to_tags[idx][tag_to_idx[data['tag']]] = {'weight': data['weight'], 'spoiler': data['spoiler_score'] }

In [14]:
#Build tags for user based on tags of top 10 vns rated by user 
t10_vns_per_user = (
    train_set.sort_values(["user_id", "vote"], ascending=[True, False])
    .groupby("user_id")
    .head(10)
)

In [15]:
user_idx_to_tags = defaultdict(lambda:defaultdict(int))
#We'll go with cumulative weight per tag instead of mean weight per tag for users as this allows tag weights to better encode strong consistent user preferences for a given tag
for _, row in t10_vns_per_user.iterrows():
   #For each user get all tags from top 10 vns
   vn_tags = vn_idx_to_tags[row['vn_idx']]
   for tag, data in vn_tags.items():
      user_idx_to_tags[row['user_idx']][tag] += data['weight']

In [16]:
#Pickling tag data for use with the model.
import pickle 

file_path = "vn_tags.pickle"
with open(file_path, "wb") as f:
    pickle.dump(dict(vn_idx_to_tags), f)

file_path = "user_tags.pickle"
with open(file_path, "wb") as f:
    pickle.dump(dict(user_idx_to_tags), f)
    
file_path = "tags.pickle"
with open(file_path, "wb") as f:
    pickle.dump(dict(tag_to_idx), f)
